# Phase 4 — RAG with ChromaDB

## Purpose
This notebook demonstrates **Phase 4** of the AI-Powered Loan Origination Copilot: replacing hallucinated policy answers with retrieval-augmented generation (RAG) over the bank's own policy documents stored in ChromaDB.

## What this notebook illustrates
| # | Demo | What it proves |
|---|------|----------------|
| 1 | Retrieval Inspection | ChromaDB returns relevant chunks for loan queries |
| 2 | Before/After — Q5 (Interest Rate) | Phase 3 may hallucinate; Phase 4 cites the policy document |
| 3 | Before/After — Q2 (Document Checklist) | Phase 4 lists product-specific docs from the policy file |
| 4 | Policy-Grounded Multi-Turn | NRI eligibility, prepayment charges, top-up — accurate from policy |
| 5 | Langfuse Trace Verification | RAG-augmented traces visible in Langfuse |

It documents **two Phase 4 limitations** — no deterministic tools, no multi-turn profile collection — that motivate Phase 5 (ReAct executor with tools).

## What you will achieve by running it
1. ChromaDB index built from 5 policy documents (~160 chunks)
2. Side-by-side evidence that Phase 4 answers are grounded in bank policy, not training data
3. Confirmed retrieval precision: top-3 chunks are relevant to the query
4. Optional: Langfuse traces for RAG evaluation

> **API key required.** Set `OPENAI_API_KEY` in Colab Secrets (🔑 sidebar) or `.env` file.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this cell first. Safe to re-run.
# Works in Google Colab and local Jupyter equally.
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, os

# 1. Install required packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'langchain', 'langchain-openai', 'langchain-community', 'langchain-core',
    'langchain-chroma', 'langchain-text-splitters',
    'python-dotenv', 'chromadb', 'openai', 'tiktoken'],
    check=True)
print('✓ Packages ready')

# 2. Detect runtime and set project root
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Capstone/code'
else:
    PROJECT_ROOT = os.path.abspath('..')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'✓ Project root: {PROJECT_ROOT}')

# 3. Load keys
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

if IN_COLAB:
    from google.colab import userdata
    for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL',
                'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass

# 4. OPENAI_API_KEY is required — prompt if missing
if not os.environ.get('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY not found — enter it now: ')
    print('✓ API key set manually')

print('✓ Environment ready')
print(f'  OPENAI_API_KEY : set')
print(f'  Langfuse keys  : {"set" if os.environ.get("LANGFUSE_PUBLIC_KEY") else "not set (optional)"}')

---
## Phase 4 — What RAG adds over Phase 3

| Dimension | Phase 3 (LLM only) | Phase 4 (LLM + RAG) |
|-----------|-------------------|---------------------|
| Policy answers | From LLM training data (may be stale / hallucinated) | From `knowledge/raw/*.txt` via ChromaDB |
| Rate/fee answers | Approximate, may differ from bank policy | Quoted verbatim from policy chunks |
| NRI / top-up queries | Generic bank knowledge | Specific to this bank's policy document |
| Index build time | N/A | ~30 seconds (one-time) |
| API key required | Yes | Yes (embeddings + LLM) |
| Tools used | None | None (Phase 5 adds ReAct tools) |

In [ ]:
# Build ChromaDB index — run once; safe to re-run (overwrites existing collection)
from retrieval.document_loader import load_documents
from retrieval.chunker import chunk_documents
from retrieval.chroma_store import build_store, load_store
import os
from deployment.config import CHROMA_DB_PATH

chroma_populated = any(os.scandir(CHROMA_DB_PATH)) if CHROMA_DB_PATH.exists() else False

if chroma_populated:
    print('ChromaDB already built — loading existing collection.')
    store = load_store()
else:
    print('Building ChromaDB index from policy documents...')
    docs = load_documents()
    print(f'  Loaded {len(docs)} documents')
    chunks = chunk_documents(docs)
    print(f'  Created {len(chunks)} chunks')
    store = build_store(chunks)
    print(f'  Persisted to: {CHROMA_DB_PATH}')

print(f'✓ Collection ready: {store._collection.name}  '
      f'({store._collection.count()} vectors)')

---
## Demo 1 — Retrieval Inspection

Before evaluating the full RAG pipeline, verify that ChromaDB returns **relevant chunks** for representative queries.
The top-3 chunks are shown for two queries — one rate question, one eligibility question.
Each chunk shows its source document and the text that will be injected into the LLM system prompt.

In [ ]:
from retrieval.retriever import retrieve

queries = [
    'home loan interest rate CIBIL 750',
    'MSME loan eligibility business owner',
    'car loan prepayment charges',
]

for q in queries:
    chunks = retrieve(q, k=3)
    print(f'Query : "{q}"')
    print(f'Chunks returned: {len(chunks)}')
    for i, c in enumerate(chunks, 1):
        print(f'  [{i}] source={c.metadata.get("source", "?")}  '
              f'chars={len(c.page_content)}')
        print(f'      {c.page_content[:200].replace(chr(10), " ")}...')
    print()

---
## Demo 2 — Before/After: Q5 (Interest Rate)

**Question:** "What is the interest rate for a home loan?"

- **Phase 3** answers from LLM training data — may quote generic industry ranges.
- **Phase 4** retrieves the bank's own policy chunk and uses it as ground truth.

The key compliance check: V3 system prompt says *never quote a specific rate* (direct to branch).  
Phase 4 should honour this while still citing the source document.

In [ ]:
from agent.core_agent import LoanCopilotAgent

q5 = 'What is the interest rate for a home loan?'

ph3 = LoanCopilotAgent(phase=3)
ph4 = LoanCopilotAgent(phase=4)

r3 = ph3.chat(q5)
r4 = ph4.chat(q5)

print('── Phase 3 (no RAG) ──')
print(r3)
print()
print('── Phase 4 (RAG-grounded) ──')
print(r4)

---
## Demo 3 — Before/After: Q2 (Document Checklist)

**Question:** "What documents do I need for a personal loan as a self-employed person?"

- **Phase 3** generates a generic list from LLM knowledge.
- **Phase 4** retrieves the specific document checklist section from the personal loan policy file.

The retrieved context should cause Phase 4 to list the exact documents defined in `personal_loan_policy.txt`.

In [ ]:
q2 = 'What documents do I need for a personal loan as a self-employed person?'

ph3 = LoanCopilotAgent(phase=3)
ph4 = LoanCopilotAgent(phase=4)

r3 = ph3.chat(q2)
r4 = ph4.chat(q2)

print('── Phase 3 (no RAG) ──')
print(r3)
print()
print('── Phase 4 (RAG-grounded) ──')
print(r4)

---
## Demo 4 — Policy-Grounded Answers (Multi-Query)

These are questions Phase 3 would answer from training data (potentially stale or wrong).  
Phase 4 should answer from the bank's own policy documents.

| Question | Why it's a RAG test |
|----------|--------------------|
| NRI eligibility for home loan | NRI-specific terms unlikely to match any generic LLM knowledge |
| MSME loan for manufacturing | Business-type restrictions that vary by bank |
| Car loan prepayment | Fee structure is bank-specific |
| Personal loan FOIR limit | Bank-specific limit may differ from industry average |

In [ ]:
policy_queries = [
    'Can an NRI apply for a home loan?',
    'Is a manufacturing business eligible for an MSME loan?',
    'Are there prepayment charges on a car loan?',
    'What is the FOIR limit for a personal loan?',
]

agent = LoanCopilotAgent(phase=4)

for q in policy_queries:
    agent.reset()
    r = agent.chat(q)
    print(f'Q: {q}')
    print(f'A: {r}')
    print()

---
## Phase 4 Limitations

These motivate Phase 5 (ReAct executor with tools).

### Limitation 1 — No Deterministic Tools

Phase 4 still has no `check_eligibility`, `calculate_emi`, or `get_document_checklist` tools.
Eligibility decisions and EMI figures are generated by the LLM — values may vary across runs and are not computed by the proven formulas from Phase 2.

**Fix in Phase 5:** ReAct executor wires all 5 tools. The LLM decides *when* to call them; the tools return deterministic results.

### Limitation 2 — RAG Context Window

Each turn retrieves a fresh top-3 based on that turn's query. If the user asks a follow-up question using pronouns ("what about for self-employed?"), the retriever gets a vague query and may return less relevant chunks.

**Fix in Phase 5:** The ReAct agent maintains conversation context and can re-phrase retrieval queries using prior turns.

In [ ]:
# Demonstrate Limitation 1: Phase 4 LLM calculates EMI — result may not match Phase 2 formula
agent = LoanCopilotAgent(phase=4)
agent.reset()

print('Demonstrating Limitation 1 — EMI from LLM vs Phase 2 deterministic formula:')
print()

from tools.emi_calculator import calculate_emi
deterministic = calculate_emi.invoke({
    'principal': 3000000,
    'annual_rate_percent': 9.0,
    'tenure_months': 240,
})
print(f'Phase 2 deterministic EMI (₹30L @ 9% × 240 mo): ₹{deterministic["emi"]:,.0f}/month')
print()

r = agent.chat('What is the EMI for a 30 lakh home loan at 9% for 20 years?')
print(f'Phase 4 LLM response:')
print(r)
print()
print('NOTE: Phase 4 LLM may estimate EMI differently — Phase 5 tools will always be exact.')

---
## Demo 5 — Langfuse Trace Verification

Each `agent.chat()` call in Demos 2–4 creates a Langfuse trace when keys are configured.  
The RAG context is injected into the system prompt — it is visible in the trace's prompt section.

To enable: add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST` to Colab Secrets or `.env`.

In [ ]:
from deployment.config import LANGFUSE_PUBLIC_KEY, LANGFUSE_HOST
from monitoring.langfuse_logger import flush

if LANGFUSE_PUBLIC_KEY:
    flush()
    ph4_agent = LoanCopilotAgent(phase=4)
    print(f'✓ Langfuse events flushed → {LANGFUSE_HOST}')
    print(f'  Dashboard: {LANGFUSE_HOST} → Traces → filter by tag phase_4')
    print(f'  Session ID: {ph4_agent.session_id}')
    print()
    print('In each trace, the System prompt section shows the retrieved policy chunks')
    print('injected as [RETRIEVED POLICY CONTEXT] — confirming RAG is active.')
else:
    print('Langfuse not configured.')
    print('Add LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY to Colab Secrets or .env to enable tracing.')